# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which, from the repository root, you can run this via:

```bash
jupyter nbconvert --to notebook --execute _scripts/talkmap.ipynb --output talkmap_out.ipynb --output-dir=_scripts --ExecutePreprocessor.cwd=.
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [ ]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import os
import shutil
import re
import time
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

# Newer versions of nbconvert/nbclient start the kernel in the notebook's
# own directory (_scripts/) and ignore --ExecutePreprocessor.cwd=. All paths
# in this notebook are relative to the repository root, so normalise here.
if os.path.basename(os.getcwd()) == '_scripts':
    os.chdir('..')

In [ ]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Nominatim's usage policy caps requests at 1/second
RATE_LIMIT_SECONDS = 1

# Manual corrections for locations Nominatim can't resolve well on its own.
# geocode_with_fallback() strips down to a less specific query on failure,
# which can land on something technically valid but far too coarse -- e.g.
# "Casa Matemática Oaxaca, Mexico" fails outright, falls back all the way to
# just "Mexico", and geocodes to the whole country's centroid instead of the
# actual city of Oaxaca.
GEOCODE_OVERRIDES = {
    "Casa Matemática Oaxaca, Mexico": "Oaxaca, Mexico",
}

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""


def geocode_with_fallback(query, timeout):
    """Geocode, falling back to a shorter (less specific) query if no match is found."""
    time.sleep(RATE_LIMIT_SECONDS)
    result = geocoder.geocode(query, timeout=timeout)
    if result is None and "," in query:
        return geocode_with_fallback(query.split(",", 1)[1].strip(), timeout)
    return result

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description: title, month & year, exact city, and the
    # conference/institute (venue covers both "it was a conference" and
    # "for invited talks, the institute")
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    city = data.get('city', location.split(',')[0]).strip()
    country = location.rsplit(',', 1)[-1].strip()
    month_year = data['date'].strftime('%b %Y')
    description = f"<strong>{title}</strong><br>{month_year}<br>{venue}<br>{city}, {country}"
    if data.get('online'):
        description += "<br><em>(online)</em>"

    # Strip trailing "(online)"-style annotations before geocoding
    geocode_query = re.sub(r"\s*\([^)]*\)\s*$", "", location)
    geocode_query = GEOCODE_OVERRIDES.get(location, geocode_query)

    # Geocode the location and report the status
    try:
        result = geocode_with_fallback(geocode_query, TIMEOUT)
        if result is None:
            print(f"Warning: no geocode match found for {geocode_query}, skipping pin")
            continue
        location_dict[description] = result
        print(description, result)
    except ValueError as ex:
        print(f"Error: geocode failed on input {geocode_query} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {geocode_query} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {geocode_query} with message {ex}")

In [ ]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="_talkmap", hashed_usernames=False)

# getorg writes its own default map.html/screen.css (Mercator tiles,
# fixed 800x600 box) on every run; overwrite them with our customized
# versions (single non-repeating image basemap, capped zoom, no grey box).
shutil.copyfile("_scripts/talkmap_assets/map.html", "_talkmap/map.html")
shutil.copyfile("_scripts/talkmap_assets/screen.css", "_talkmap/leaflet_dist/screen.css")
shutil.copyfile("_scripts/talkmap_assets/countries.geojson", "_talkmap/countries.geojson")